# Train model nhận diện quả — Nhandienqua (Google Colab)

**Trước khi chạy:**
1. Trên máy: `python scripts/pack_for_colab.py` → `vision/colab/ndq_quality.zip`, `ndq_detect.zip`.
2. Tải 2 file zip lên Google Drive vào thư mục **`MyDrive/nhandienqua/`**.
3. Colab: *Runtime → Change runtime type → GPU (T4)*, rồi *Run all*.

Checkpoint lưu thẳng vào Drive (`MyDrive/nhandienqua/runs/`). Nếu Colab ngắt giữa chừng, chạy lại notebook — ô train tự **tiếp tục** từ `last.pt`.

Tham số giống hệt `vision/scripts/train.py` (YOLO26n, Ultralytics 8.4.174).

**Sau khi xong:** tải `MyDrive/nhandienqua/weights/fruit_quality.pt` và `fruit_det.pt` về `vision/weights/` trên máy, rồi chạy `scripts/evaluate.py` và `scripts/export.py`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/nhandienqua'
!pip -q install ultralytics==8.4.174

In [ ]:
# Giải nén dữ liệu ra ổ cục bộ của Colab (đọc nhanh hơn nhiều so với đọc trên Drive)
import os, shutil, zipfile, yaml
os.makedirs('/content/data', exist_ok=True)
for name in ['quality', 'detect']:
    if not os.path.exists(f'/content/data/{name}'):
        shutil.copy(f'{DRIVE}/ndq_{name}.zip', '/content/')
        zipfile.ZipFile(f'/content/ndq_{name}.zip').extractall('/content/data')
        os.remove(f'/content/ndq_{name}.zip')
# data.yaml mang đường dẫn của máy Windows → ghi lại cho Colab
for y in ['data.yaml', 'data_synth_test.yaml', 'data_real_test.yaml']:
    p = f'/content/data/detect/{y}'
    d = yaml.safe_load(open(p, encoding='utf-8'))
    d['path'] = '/content/data/detect'
    yaml.safe_dump(d, open(p, 'w', encoding='utf-8'), allow_unicode=True)
for s in ['train', 'val', 'test']:
    n = sum(len(f) for _, _, f in os.walk(f'/content/data/quality/{s}'))
    print('quality', s, n, 'ảnh')
for s in ['train', 'val', 'test_real', 'test_synth']:
    print('detect', s, len(os.listdir(f'/content/data/detect/images/{s}')), 'ảnh')

In [ ]:
from pathlib import Path
from ultralytics import YOLO
RUNS = f'{DRIVE}/runs'
os.makedirs(f'{DRIVE}/weights', exist_ok=True)

def train_or_resume(task, init, out_name, **kw):
    run = Path(RUNS) / task / 'train'
    last = run / 'weights' / 'last.pt'
    if last.exists():
        print('Tiếp tục từ', last)
        try:
            YOLO(str(last)).train(resume=True)
        except Exception as e:  # lần chạy trước đã train xong → không còn gì để tiếp tục
            print('Bỏ qua resume:', e)
    else:
        YOLO(init).train(project=f'{RUNS}/{task}', name='train', exist_ok=True, seed=0, plots=True, workers=2, **kw)
    shutil.copy(run / 'weights' / 'best.pt', f'{DRIVE}/weights/{out_name}')
    print('Đã lưu', f'{DRIVE}/weights/{out_name}')

## 1. Model tốt / hư (YOLO26n-cls, 224 px) — khoảng 20–40 phút trên T4

In [ ]:
train_or_resume('quality', 'yolo26n-cls.pt', 'fruit_quality.pt',
                data='/content/data/quality', imgsz=224, epochs=30, batch=128, patience=8,
                fliplr=0.5, flipud=0.5)

## 2. Model phát hiện quả (YOLO26n, 640 px) — khoảng 2,5–4 giờ trên T4
Bản Colab miễn phí có thể bị ngắt: chạy lại notebook để tiếp tục.

In [ ]:
train_or_resume('detect', 'yolo26n.pt', 'fruit_det.pt',
                data='/content/data/detect/data.yaml', imgsz=640, epochs=60, batch=32, patience=15,
                fliplr=0.5, flipud=0.5, degrees=10, mosaic=1.0, close_mosaic=10)

## 3. Kiểm tra nhanh (số liệu chính thức chạy `scripts/evaluate.py` trên máy)

In [ ]:
det = YOLO(f'{DRIVE}/weights/fruit_det.pt')
for y in ['data_real_test.yaml', 'data_synth_test.yaml']:
    m = det.val(data=f'/content/data/detect/{y}', split='val', imgsz=640, verbose=False, plots=False)
    print(y, 'mAP50 =', round(m.box.map50, 3), 'mAP50-95 =', round(m.box.map, 3))
q = YOLO(f'{DRIVE}/weights/fruit_quality.pt').val(data='/content/data/quality', split='test', imgsz=224, verbose=False, plots=False)
print('quality top-1 (test) =', round(q.top1, 3))